# Ajustar pop-up de horários + diagnóstico — Mobilidade Urbana — v1
- Faz backup do web map **Mobilidade Urbana** e atualiza o pop-up das linhas: hora em **HH:MM**, nomes dos dias com acento, **ponto de saída** antes dos horários.
- Diagnóstico (só leitura): rotas sem horários, horários sem rota, valores de `dia_tipo` e variantes com possível erro de digitação (ex.: "Bairri").
Saída: `Documents\Saidas_Notebooks\Ajustar_Popup_Mobilidade_v1.txt`

In [ ]:
# ===== CÉLULA 1 — Configuração e conexão =====
import os, json, datetime, collections
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, Table
NB_NOME = "Ajustar_Popup_Mobilidade_v1"
URL_APUC = "https://geo.apucarana.pr.gov.br/portal"
WEBMAP_ID = "c80cb05e18a645609f8f84740078b859"
HORARIOS_ID = "7b643a7396af425d820da4c55b06effe"
URL_LINHAS = "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Linhas_Onibus/FeatureServer/0"
URL_HORARIOS = "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Mobilidade_Horarios_Onibus/FeatureServer/0"
TITULO_CAMADA = "Linhas de ônibus"

home = os.path.expanduser("~")
_c = [os.path.join(home, "Documents", "Saidas_Notebooks"), os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _c if os.path.isdir(c)), _c[0]); os.makedirs(PASTA_SAIDA, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f: f.write(s + "\n")

log("▶ CÉLULA 1 — Configuração e conexão: iniciando...")
gis = GIS("pro")
log("  Conectado como:", gis.users.me.username, "|", gis.url)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Diagnóstico dos dados (somente leitura) =====
log("\n▶ CÉLULA 2 — Diagnóstico: iniciando...")
lin = FeatureLayer(URL_LINHAS, gis).query(where="1=1", out_fields="id_rota,cod_linha,nome,sentido,sentido_variante", return_geometry=False).features
hor = Table(URL_HORARIOS, gis).query(where="1=1", out_fields="id_rota,dia_tipo,ponto,horario", return_geometry=False).features
L = [f.attributes for f in lin]; H = [f.attributes for f in hor]
rotas_lin = {a["id_rota"] for a in L}
cont_hor = collections.Counter(a["id_rota"] for a in H)

sem = [a for a in L if cont_hor.get(a["id_rota"], 0) == 0]
log(f"\n  Rotas (feições de linha): {len(L)} | com horários: {len(L)-len(sem)} | SEM horários: {len(sem)}")
for a in sorted(sem, key=lambda x: (str(x['cod_linha']), str(x['id_rota']))):
    log(f"   - {a['id_rota']:<14} linha {a['cod_linha']} {a['nome']} | sentido: {a['sentido']} | variante: {a['sentido_variante']}")

orf = sorted(set(cont_hor) - rotas_lin, key=str)
log(f"\n  id_rota da tabela de horários SEM linha no mapa: {len(orf)}")
for r in orf:
    log(f"   - {r} ({cont_hor[r]} horários)")

log("\n  Valores de dia_tipo:", dict(collections.Counter(a["dia_tipo"] for a in H)))
log("  Valores de sentido_variante:", dict(collections.Counter(a["sentido_variante"] for a in L)))
log("  Valores de sentido:", dict(collections.Counter(a["sentido"] for a in L)))
amostra = [a["horario"] for a in H[:5]]
log("  Amostra do campo horario:", amostra, "| tipo:", type(H[0]["horario"]).__name__ if H else "-")
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Backup e atualização do pop-up =====
log("\n▶ CÉLULA 3 — Atualização do pop-up das linhas: iniciando...")
ARCADE = """
var tab = FeatureSetByPortalItem(Portal('%s'), '%s', 0, ['id_rota','dia_tipo','ponto','horario'], false);
var rota = $feature.id_rota;
var hs = Filter(tab, 'id_rota = @rota');
if (Count(hs) == 0) {
  return { type: 'text', text: '<p style="color:#888;margin:4px 0">Sem horários cadastrados para esta rota.</p>' };
}
var nomes = { 'Dias Uteis': 'Dias úteis', 'Dias uteis': 'Dias úteis', 'DIAS UTEIS': 'Dias úteis', 'Uteis': 'Dias úteis',
              'Sabado': 'Sábado', 'Sabados': 'Sábados', 'SABADO': 'Sábado',
              'Domingo': 'Domingo', 'Domingos': 'Domingos', 'Domingos e Feriados': 'Domingos e feriados',
              'Domingo e Feriado': 'Domingo e feriado', 'Feriados': 'Feriados' };
var html = '<div style="font-family:Arial,sans-serif;font-size:13px">';
for (var d in Distinct(hs, 'dia_tipo')) {
  var dia = d.dia_tipo;
  var rotulo = IIf(HasKey(nomes, dia), nomes[dia], dia);
  html += '<div style="margin:12px 0 4px;font-weight:bold;color:#1A2942;border-bottom:2px solid #F5A623">' + rotulo + '</div>';
  var sub = OrderBy(Filter(hs, 'dia_tipo = @dia'), 'ponto ASC, horario ASC');
  var pontoAnt = '#';
  for (var h in sub) {
    var p = IIf(IsEmpty(h.ponto), '', Text(h.ponto));
    if (p != pontoAnt) {
      if (pontoAnt != '#') { html += '</div>'; }
      if (p != '') { html += '<div style="font-size:11px;color:#555;margin:6px 0 2px">&#128205; Saída: ' + p + '</div>'; }
      html += '<div>';
      pontoAnt = p;
    }
    var hr = Left(Text(h.horario), 5);
    html += '<span style="display:inline-block;margin:2px;padding:3px 8px;border-radius:4px;background:#1A2942;color:#fff;font-weight:bold">' + hr + '</span>';
  }
  html += '</div>';
}
html += '</div>';
return { type: 'text', text: html };
""" % (URL_APUC, HORARIOS_ID)

wm_item = gis.content.get(WEBMAP_ID)
wm = wm_item.get_data()
bkp = os.path.join(PASTA_SAIDA, f"backup_webmap_mobilidade_{datetime.datetime.now():%Y%m%d_%H%M%S}.json")
with open(bkp, "w", encoding="utf-8") as f:
    json.dump(wm, f, ensure_ascii=False, indent=2)
log("  Backup do web map:", bkp)

alvo = [ol for ol in wm.get("operationalLayers", []) if ol.get("title") == TITULO_CAMADA]
if not alvo:
    raise RuntimeError(f"Camada '{TITULO_CAMADA}' não encontrada no web map.")
pop = alvo[0].setdefault("popupInfo", {})
info = {"name": "horarios", "title": "Horários", "expression": ARCADE, "returnType": "dictionary"}
pop["expressionInfos"] = [info]
pop["popupElements"] = [{"type": "fields"}, {"type": "expression", "expressionInfo": info}]
wm_item.update(item_properties={"text": json.dumps(wm)})
log("  Pop-up atualizado: hora HH:MM, dias com acento, ponto de saída.")
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Links =====
log("\n▶ CÉLULA 4 — Links: iniciando...")
log("  Web map (Map Viewer):", f"{URL_APUC}/apps/mapviewer/index.html?webmap={WEBMAP_ID}")
log("  Item do web map:", f"{URL_APUC}/home/item.html?id={WEBMAP_ID}")
log("  Dica: no Map Viewer, recarregue com Ctrl+F5 antes de clicar numa linha.")
log(f"  Saída completa: {ARQ_TXT}")
log("✔ CÉLULA 4 concluída com sucesso.")
